# H&M 비즈니스 인사이트 분석 및 연관 분석
> **목적**: 전처리된 데이터를 바탕으로 단품 구매 현상을 진단하고 교차 판매 전략을 수립합니다.

### 주요 작업 내용:
1. **EDA**: 연령대별 고객 분포 및 인기 카테고리 시각화
2. **문제 정의**: 85.2% 단품 구매율 진단 및 해결책 모색
3. **연관 분석**: `itertools`를 활용한 연령대별 상품 조합(Pair) 추출
4. **전략 제안**: 데이터 기반의 타겟별 마케팅 가이드라인 도출

In [ ]:
# Step 0. 라이브러리 임포트 및 시각화 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 전처리 완료된 데이터
df_clean = pd.read_pickle('hm_cleaned_data.pkl')


Step 1. 탐색적 데이터 분석(EDA): 시각화를 통해 20대와 30대 고객이 주력 소비층임을 파악하고 전체적인 제품 판매 비중을 확인

In [ ]:
# Step 1. 탐색적 데이터 분석 (EDA)

print("--- [Step 1] 탐색적 데이터 분석(EDA) 시작 ---")

# 1.1 연령대별 고객 분포
plt.figure(figsize=(10, 6))
sns.countplot(data=df_clean, x='age_group', palette='pastel', order=['16-24', '25-34', '35+'])
plt.title('연령대별 구매 고객 분포')
plt.show()

# 1.2 카테고리별 판매량 TOP 10
top_cats = df_clean['product_group_name'].value_counts().head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, palette='viridis')
plt.title('인기 제품 카테고리 TOP 10')
plt.show()

Step 2. 문제 정의: 85.2%의 단품 구매 비중을 실제 데이터로 산출하여 문제의 심각성을 입증

In [ ]:
# Step 2. 비즈니스 문제 정의: 단품 구매 현상 (85.2%)

print("\n--- [Step 2] 비즈니스 문제 정의: Basket Size 분석 ---")

# 장바구니 크기(Basket Size) 계산: 동일 고객, 동일 일자 기준
basket_size = df_clean.groupby(['customer_id', 't_dat']).size().reset_index(name='count')

# 단품 구매 비중 산출
single_rate = (basket_size[basket_size['count'] == 1].shape[0] / len(basket_size)) * 100
print(f"실제 데이터 확인 결과 단품 구매(Basket Size=1) 비중: {single_rate:.1f}%")

Step 3. 상품 연관 분석: itertools.combinations를 활용해 2개 이상 구매 고객의 장바구니 내에서 동시 구매 상품 쌍(Pair)을 추출

In [ ]:
# Step 3. 핵심 분석: 연령대별 연관 상품 조합 추출

print("\n--- [Step 3] 연령대별 연관 상품 쌍(Pair) 추출 로직 실행 ---")

# 다품 구매 장바구니 필터링 (장바구니 크기 > 1)
multi_basket = basket_size[basket_size['count'] > 1]
multi_transactions = df_clean.merge(multi_basket[['customer_id', 't_dat']], on=['customer_id', 't_dat'], how='inner')

def get_top_pairs(df_part, age_group):
    """특정 연령대의 장바구니 내 상품 조합 빈도 추출"""
    group_df = df_part[df_part['age_group'] == age_group]
    # 장바구니별 상품 리스트 생성
    baskets = group_df.groupby(['customer_id', 't_dat'])['prod_name'].apply(list)
    
    pair_counts = {}
    for items in baskets:
        # 중복 제거 및 정렬 후 모든 가능한 2개 조합 생성
        for pair in combinations(sorted(set(items)), 2):
            pair_counts[pair] = pair_counts.get(pair, 0) + 1
            
    pair_df = pd.DataFrame([{'pair': f"{p[0]} & {p[1]}", 'count': c} for p, c in pair_counts.items()])
    return pair_df.sort_values('count', ascending=False).head(10)

Step 4. 결과 시각화 및 제안: 연령대별 인기 상품 조합 TOP 10을 도출하여, 비키니 세트 등 데이터 기반의 교차 판매(Cross-selling) 마케팅 전략을 수립

In [ ]:
# Step 4. 최종 결과 시각화 및 전략 도출

print("\n--- [Step 4] 분석 결과 시각화 및 마케팅 전략 도출 ---")

age_groups = ['16-24', '25-34', '35+']
for age in age_groups:
    top_10 = get_top_pairs(multi_transactions, age)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=top_10, x='count', y='pair', palette='flare')
    plt.title(f'{age} 연령대 인기 상품 조합 (교차 판매 타겟)')
    plt.show()

print("\n모든 비즈니스 분석 과정이 완료되었습니다.")